# Sentinel-2 L2A: mosaicos y clasificación de fuego

Prepare Colab → cargue vectores → inspeccione GEE → descargue y compare → dibuje muestras → clasifique con RF.

Edite los parámetros aquí; la lógica está en `Sentinel2_pipeline_test.py`.
Las escenas se procesan en `/content/sentinel2_temporal`; el mosaico terminado se conserva en Drive.

## 1. Preparar Colab

Ejecute la instalación y cargue `Sentinel2_pipeline_test.py` cuando se solicite.
En local, use `uv sync --locked` y el núcleo de `.venv`.

In [ ]:
try:
    from google.colab import files
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

if EN_COLAB:
    %pip install -q gdown "geopandas>=1.1" "pystac-client>=0.9" "rasterio>=1.4" "folium>=0.20" matplotlib tqdm planetary-computer earthengine-api geemap scikit-learn psutil google-api-python-client "ipywidgets>=8,<9" "jupyter-loopback[comm]>=0.3,<1"

In [ ]:
from pathlib import Path
import sys
from importlib import import_module, reload

CARPETA_MODULO = Path("/content") if EN_COLAB else Path("/Users/lujem/Documents/GAIA/Scripts/MBCOLOMBIA-GITHUB/colombia-fire/mapbiomas_fire_monitor/version_01/src/core")
if EN_COLAB and not (CARPETA_MODULO / "Sentinel2_pipeline_test.py").is_file():
    print("Cargue Sentinel2_pipeline_test.py.")
    (CARPETA_MODULO / "Sentinel2_pipeline_test.py").write_bytes(files.upload()["Sentinel2_pipeline_test.py"])
sys.path.insert(0, str(CARPETA_MODULO))
NotebookWorkflow = reload(import_module("Sentinel2_pipeline_test")).NotebookWorkflow

DIRECTORIO_TRABAJO = Path("/content/sentinel2_temporal") if EN_COLAB else None
workflow = NotebookWorkflow(DIRECTORIO_TRABAJO)


## 2. Shapefiles de Drive

Los IDs incluyen el índice de teselas y las 26 regiones de monitoreo.
Active `AUTENTICAR_DRIVE` si los archivos son privados.

In [ ]:
TILES_ID = "1sJlyT9TfGx2Qb6gW4ptupKsPnc-oa39d"
REGIONS_ID = "16s0_U649udFVTXGRwHHjsHbHFv3WKD98"
AUTENTICAR_DRIVE = True  #@param {type:"boolean"}

workflow.load_vectors(TILES_ID, REGIONS_ID, authenticate=AUTENTICAR_DRIVE)

DRIVE_ROOT = "/content/drive/MyDrive/FuegoMonitor"
if EN_COLAB:
    workflow.use_drive(DRIVE_ROOT)


## Inspeccionar en GEE

Edite `PARAMETERS` y autentique su cuenta. La fecha final es **exclusiva**.
Las bandas y la máscara SCL se comparten con la descarga.

GEE muestra el mosaico según el zoom; `resolution_m` controla la salida local.
Active `MOSTRAR_ESTADISTICAS_GEE` para consultar conteos y fechas.

In [ ]:
# Lista única de parámetros: GEE y descarga usan estos mismos valores.
PARAMETERS = {
    "project": "mapbiomas-fire-485203",
    "regions_asset": "projects/mapbiomas-colombia/assets/FIRE/AUXILIARY_DATA/regiones_monitor_fuego",
    "tiles_asset": "projects/mapbiomas-colombia/assets/DATOS_AUXILIARES/VECTORES/SENTINEL/sentinel_2_index_shapefile",
    "region_id": 7,
    "start_date": "2026-01-01",
    "end_date": "2026-02-01",  # exclusiva: incluye todo enero, no el 1 de febrero
    "max_cloud_cover": 20,
    "bands": ["B11", "B8", "B4", "B12"],  # amplíe esta lista para descargar más bandas
    "resolution_m": 20,  # resolución del mosaico descargado
    "scl_exclude": [0, 1, 2, 3, 8, 9, 10, 11],
    "vis": {"bands": ["swir1", "nir", "red"], "min": 0.03, "max": 0.4},
}
# Rojo, NIR, SWIR1, SWIR2 se incluyen siempre para los índices; SCL es auxiliar.


In [ ]:
MOSTRAR_ESTADISTICAS_GEE = True  #@param {type:"boolean"}

Map = workflow.preview(PARAMETERS, include_metadata=MOSTRAR_ESTADISTICAS_GEE)
Map


## 5. Descargar y componer

Active `DESCARGAR_IMAGENES`. Se reutiliza el mosaico guardado para las mismas fechas, región, fuente y parámetros.
`SOBRESCRIBIR_MOSAICO = True` fuerza una nueva descarga y composición.

Se procesa temporalmente y se guarda el COG, metadatos y tiempos en `DRIVE_ROOT/mosaics/AAAA-MM`.
`CONSERVAR_IMAGENES = False` elimina las escenas tras procesarlas.
Los tiempos de descarga y procesamiento se solapan.


In [ ]:
SOURCE = "planetary-computer"  #@param ["planetary-computer", "earth-search"]
DESCARGAR_IMAGENES = True  #@param {type:"boolean"}
CONSERVAR_IMAGENES = False  #@param {type:"boolean"}

SOBRESCRIBIR_MOSAICO = False  #@param {type:"boolean"}

composicion = workflow.download(
    PARAMETERS, source=SOURCE, enabled=DESCARGAR_IMAGENES,
    keep_inputs=CONSERVAR_IMAGENES, overwrite=SOBRESCRIBIR_MOSAICO,
)


## 6. Comparación pausada

La comparación está comentada. Continúe en la sección 7 para abrir el mosaico y tomar muestras.


In [ ]:
# VIEW = "color"  #@param ["color", "NBR"]
# RENOVAR_TESELAS_GEE = False  #@param {type:"boolean"}

# comparison = workflow.compare(
#     PARAMETERS, source=SOURCE, view=VIEW, refresh_gee=RENOVAR_TESELAS_GEE,
# )
# comparison

## 7. Muestras, modelos y clasificación

Edite muestras en Folium y elija el archivo para entrenar.
Seleccione **Crear modelo nuevo** y el algoritmo, o aplique/reentrene uno guardado.
Los resultados quedan como capas independientes; el panel muestra tiempo y RAM.


In [ ]:
MODEL_PARAMETERS = {
    "gaussian_nb": {"var_smoothing": 1e-9},
    "logistic_regression": {"C": 1.0, "max_iter": 1000},
    "svm": {"C": 1.0, "gamma": "scale"},
    "extra_trees": {},
    "gradient_boosting": {"learning_rate": 0.1, "max_depth": 3},
}
RF_PARAMETERS = {
    "classifier_parameters": MODEL_PARAMETERS,
    "n_estimators": 200, "max_depth": 20,
    "min_samples_leaf": 2,
    "max_samples_per_class": 5000, "min_samples_per_class": 10,
    "random_state": 42, "n_jobs": 2, "block_size": 512,
}


In [ ]:
MOSAIC_PATH = ""  # vacío: recuperar el mosaico de PARAMETERS
SOURCE = globals().get("SOURCE", "planetary-computer")

training = workflow.training_canvas(
    PARAMETERS, source=SOURCE, mosaic_path=MOSAIC_PATH or None, rf_parameters=RF_PARAMETERS,
)
training.show()


## Fuentes Sentinel-2 L2A

Reflectancia de superficie ortorrectificada y corregida atmosféricamente.
Cuotas consultadas el **19 de octubre de 2026**.

| Fuente | Cuota gratuita | Requisitos |
| --- | --- | --- |
| [Earth Search / AWS — COG de Nivel 2A](https://registry.opendata.aws/sentinel-2-l2a-cogs/) **(opción de descarga)** | Acceso gratuito; la documentación no especifica una cuota mensual en GB. | Ninguno para descargar COG públicos por HTTPS. |
| [Copernicus Data Space — productos de Nivel 2A](https://documentation.dataspace.copernicus.eu/Data/SentinelMissions/Sentinel2.html#sentinel-2-level-2a-surface-reflectance) | [12 TB por cada período móvil de 30 días a velocidad normal; después se reduce la velocidad.](https://documentation.dataspace.copernicus.eu/Quotas.html) | Cuenta gratuita de CDSE; token de acceso o credenciales S3. |
| [CDSE Sentinel Hub — API de procesamiento L2A](https://documentation.dataspace.copernicus.eu/APIs/SentinelHub/Data/S2L2A.html) | [10.000 unidades de procesamiento y 10.000 solicitudes al mes.](https://documentation.dataspace.copernicus.eu/Quotas.html) | Cuenta CDSE y cliente OAuth; solicitar bandas espectrales en unidades REFLECTANCE. |
| [Microsoft Planetary Computer — sentinel-2-l2a](https://planetarycomputer.microsoft.com/dataset/sentinel-2-l2a) | [Acceso gratuito; no se especifica una cuota mensual en GB; hay límites de frecuencia al solicitar tokens.](https://planetarycomputer.microsoft.com/docs/concepts/sas/) | URL firmadas con SAS; se pueden obtener tokens de forma anónima. |
| [Google Earth Engine — S2_SR_HARMONIZED](https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S2_SR_HARMONIZED) | [Nivel Comunidad: 150 horas-EECU por proyecto y mes; rendimiento reducido al superar la cuota.](https://developers.google.com/earth-engine/guides/noncommercial_tiers) | Cuenta Google, proyecto de Cloud registrado, uso no comercial verificado y autenticación. |

Descarga: Earth Search o Planetary Computer. Inspección: GEE.
Las cuotas de descarga y cómputo son distintas; una cuota no publicada no implica acceso ilimitado.